# Using Open Source LLMs Natively

Here we will see briefly how you can use popular open source LLM APIs including

- Hugging Face Transformers
- Hugging Face Serverless Inference APIs
- Hugging Face Inference Client
- Groq Cloud

## Install Dependencies

In [5]:
!pip install torch

   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/114.5 MB ? eta -:--:--
   ---------------------------------------- 1.0/114.5 MB 3.1 MB/s eta 0:00:37
    --------------------------------------- 1.6/114.5 MB 3.0 MB/s eta 0:00:38
    --------------------------------------- 2.4/114.5 MB 3.1 MB/s eta 0:00:36
   - -------------------------------------- 3.1/114.5 MB 3.1 MB/s eta 0:00:36
   - -------------------------------------- 3.9/114.5 MB 3.3 MB/s eta 0:00:34
   - -------------------------------------- 4.7/114.5 MB 3.4 MB/s eta 0:00:33
   - -------------------------------------- 5.5/114.5 MB 3.4 MB/s eta 0:00:33
   -- ------------------------------------- 6.3/114.5 MB 3.4 MB/s eta 0:00:32
   -- ------------------------------------- 7.3/114.5 MB 3.5 MB/s eta 0:00:31
   -- ------------------------------------- 8.1/114.5 MB 3.6 MB/s eta 0:00:30
   --- ------------------------------------ 9.2/114.5 MB 3.7 MB/s eta 0:00:29


In [1]:
!pip install transformers==4.53.2
!pip install accelerate==1.9.0 # useful when using models with GPUs locally via huggingface
!pip install groq==0.30.0

  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   --- ------------------------------------ 1.0/10.8 MB 6.3 MB/s eta 0:00:02
   ------- -------------------------------- 2.1/10.8 MB 5.3 MB/s eta 0:00:02
   ---------- ----------------------------- 2.9/10.8 MB 4.5 MB/s eta 0:00:02
   ------------- -------------------------- 3.7/10.8 MB 4.4 MB/s eta 0:00:02
   ---------------- ----------------------- 4.5/10.8 MB 4.3 MB/s eta 0:00:02
   ------------------- -------------------- 5.2/10.8 MB 4.2 MB/s eta 0:00:02
   ----------------------- ---------------- 6.3/10.8 MB 4.2 MB/s eta 0:00:02
   -------------------------- ------------- 7.1/10.8 MB 4.2 MB/s eta 0:00:01
   ----------------------------- ---------- 7.9/10.8 MB 4.2 MB/s eta 0:00:01
   --------------------------------- ------ 9.2/10.8 MB 4.3 MB/s eta 0:00:01
   ------------------------------------ --- 10.0/10.8 MB 4.3 MB/s eta 0:00:01
   ---------

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


## Get Hugging Face Access Token

Here you need to get an access token to be able to download or access models using Hugging Face's platform:

- Hugging Face Access Token: Go [here](https://huggingface.co/settings/tokens) and create a key with write permissions. You need to setup an account which is totally free of cost.


1. Go to [Settings -> Access Tokens](https://huggingface.co/settings/tokens) after creating your account and make sure to create a new access token with write permissions

![](https://i.imgur.com/dtS6tFr.png)

2. Remember to __Save__ your key somewhere safe as it will just be shown once as shown below. So copy and save it in a local secure file to use it later on. If you forget, just create a new key anytime.

![](https://i.imgur.com/NmZmpmw.png)

## Load Hugging Face Access Token


In [1]:
from getpass import getpass

hf_key = getpass("Enter your Hugging Face Access Token: ")

Enter your Hugging Face Access Token:  ········


## Configure Key in Environment


In [2]:
import os

os.environ["HF_TOKEN"] = hf_key

## Using LLMs Locally with Hugging Face

This is if you want to download LLMs locally completely and run it without the need of sending your data to any external server. Do note you would need a GPU to run any of these models as even the smaller language models are still essentially quite big.

Certain LLMs are gated like [Meta Llama 3.2 1B Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) so make sure to apply for access as shown below else you will get an error when using the model

![](https://i.imgur.com/M88MOu5.png)

## Load the LLM locally using Huggingface

In [ ]:
# One needs to use GPU to download the below weights and run the model locally (use google collab -> change runtime select the gpu available
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype=torch.bfloat16
)

In [4]:
chat = [
    { "role": "user", "content": "Explain what is Generative AI in 2 bullet points" },
]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 06 Apr 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Explain what is Generative AI in 2 bullet points<|eot_id|><|start_header_id|>assistant<|end_header_id|>




Remember to always refer to the [__documentation__](https://huggingface.co/docs/transformers/v4.18.0/en/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate) where all the arguments of the generation pipeline are mentioned in detail. Most notably:

- **max_length:** The maximum length of the sequence to be generated
- **max_new_tokens:** The maximum numbers of tokens to generate, ignore the current number of tokens. Use either max_new_tokens or max_length but not both, they serve the same purpose
- **do_sample:** Whether or not to use sampling. False means use greedy decoding i.e temperature=0
- **temperature:** Between 0 - 1, The value used to module the next token probabilities. Higher temperature means the results may vary and be more creative

In [ ]:
inputs = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt")
outputs = model.generate(input_ids=inputs.to(model.device), max_new_tokens=1000)
print(tokenizer.decode(outputs[0]))

### Pipelines make it easier to send prompts

You don't need to encode and decode your inputs and outputs everytime

In [ ]:
llama_pipe = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="cuda",
)

Device set to use cuda


In [ ]:
chat = [
    { "role": "user", "content": "Explain what is Generative AI in 2 bullet points" },
]

In [ ]:
response = llama_pipe(chat, max_new_tokens=1000)
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': [{'role': 'user', 'content': 'Explain what is Generative AI in 2 bullet points'}, {'role': 'assistant', 'content': 'Here are 2 bullet points explaining what Generative AI is:\n\n• **Artificial Intelligence that Creates Content**: Generative AI is a type of artificial intelligence that uses algorithms and machine learning to create new content, such as text, images, music, and videos, based on patterns and relationships learned from existing data. It can generate novel and coherent content that is indistinguishable from human-created work.\n\n• **Generative Models that Learn and Adapt**: Generative AI models, such as Generative Adversarial Networks (GANs) and Variational Autoencoders (VAEs), learn to generate new content by learning patterns and relationships in existing data. They can adapt to new data and generate new content that is similar to the original, but with some differences. This allows generative AI to create new content that is diverse and coherent.'}]}

In [ ]:
print(response[0]["generated_text"][-1]['content'])

Here are 2 bullet points explaining what Generative AI is:

• **Artificial Intelligence that Creates Content**: Generative AI is a type of artificial intelligence that uses algorithms and machine learning to create new content, such as text, images, music, and videos, based on patterns and relationships learned from existing data. It can generate novel and coherent content that is indistinguishable from human-created work.

• **Generative Models that Learn and Adapt**: Generative AI models, such as Generative Adversarial Networks (GANs) and Variational Autoencoders (VAEs), learn to generate new content by learning patterns and relationships in existing data. They can adapt to new data and generate new content that is similar to the original, but with some differences. This allows generative AI to create new content that is diverse and coherent.


## Using LLMs via Hugging Face Inference APIs

Thankfully HuggingFace has made its [__Inference API__](https://huggingface.co/docs/api-inference/quicktour) free to use with some basic rate limits etc. in place so you don't end up making unlimited requests on it's servers.

The best part is you can access 150,000+ deep learning models without worrying about your infrastructure.

## Load Hugging Face Access Token


In [7]:
from getpass import getpass

hf_key = getpass("Enter your Hugging Face Access Token: ")

Enter your Hugging Face Access Token:  ········


## Configure Key in Environment


In [8]:
import os

os.environ["HF_TOKEN"] = hf_key

### Create LLM API Access Function

Here we create a basic function which can access any LLM API endpoint available on HuggingFace.

For more details refer to the [detailed documentation](https://huggingface.co/docs/api-inference/detailed_parameters#text-generation-task) as needed.

In [9]:
import requests

headers = {"Authorization": "Bearer "+hf_key}

def query(payload, API_URL):
  response = requests.post(API_URL, headers=headers, json=payload)
  print('API Response:', response)
  return response.json()

## Create LLM API Access Config

Here we decide which LLMs we will access by getting their inference API endpoints.

We also set some general configuration settings. You can find the [detailed documentation](https://huggingface.co/docs/api-inference/detailed_parameters#text-generation-task) here.

Some useful config settings include:

- max_new_tokens: The amount of new tokens to be generated in the response
- do_sample: Whether or not to use sampling. False means use greedy decoding i.e temperature=0
- temperature: Between 0 - 1, The value used to module the next token probabilities. Higher temperature means the results may vary and be more creative
- return_full_text: If set to False, does not return your input prompt to the model
- wait_for_model:  If the model is not ready, wait for it instead of receiving 503. It limits the number of requests required to get your inference done
- repetition_penalty: The more a token is used within generation the more it is penalized to not be picked in successive generation passes.

In [10]:
HF_API_URL = "https://router.huggingface.co/v1/chat/completions" # updated to new endpoint as per HF changes https://huggingface.co/docs/huggingface_hub/v0.13.2/en/guides/inference
# need to mention model provider as per new HF syntax https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct?inference_api=true&inference_provider=novita&language=python&client=requests
model_name = "meta-llama/Llama-3.2-1B-Instruct:novita"
params = {
    "wait_for_model": True,
    "return_full_text": False,
    "max_new_tokens": 1000,
}


In [11]:
prompt =  "Explain what is Generative AI in 2 bullet points"
# updated prompt input format as per https://huggingface.co/docs/huggingface_hub/v0.13.2/en/guides/inference
messages = [
    {
        "role": "user",
        "content": prompt
    }
]

In [14]:
# updated payload format as per https://huggingface.co/docs/huggingface_hub/v0.13.2/en/guides/inference
output = query(payload={
                "messages": messages,
                "parameters": params,
                "model": model_name,
                },
                API_URL=HF_API_URL)

print(output['generated_text'])

API Response: <Response [400]>


KeyError: 'generated_text'

## Using LLMs via Hugging Face Inference Client

Thankfully HuggingFace has made its new [__Inference Client__](https://huggingface.co/docs/huggingface_hub/en/package_reference/inference_client) free to use with some basic rate limits etc. in place so you don't end up making unlimited requests on its servers.

The best part is you can access 150,000+ deep learning models without worrying about your infrastructure. Similar to the inference API

In [15]:
from huggingface_hub import InferenceClient

Feel free to refer to the [documentation](https://huggingface.co/docs/huggingface_hub/en/package_reference/inference_client#huggingface_hub.InferenceClient) at any time as needed for more details on function names, arguments and more.

In [16]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
client = InferenceClient(model=model_name, api_key=hf_key)

chat = [
    { "role": "user", "content": "Explain what is Generative AI in 2 bullet points" },
]

response = client.chat_completion(chat, max_tokens=1000)
print(response)

ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Here are two bullet points explaining what Generative AI is:\n\n• **Artificial Intelligence that generates new content**: Generative AI is a type of artificial intelligence (AI) that creates new content, such as images, music, text, or videos, based on patterns and algorithms learned from existing data. This AI can generate new content that is similar to, but not identical to, existing content, making it a powerful tool for creative industries, content creation, and even art.\n\n• **Unsupervised learning**: Generative AI models are typically trained using unsupervised learning, which means they are trained on large datasets without labeled or annotated data. This allows the AI to learn patterns and relationships in the data, and then use these patterns to generate new content that is similar to, but distinct from, the original data.', 

In [17]:
print(response.choices[0].message.content)

Here are two bullet points explaining what Generative AI is:

• **Artificial Intelligence that generates new content**: Generative AI is a type of artificial intelligence (AI) that creates new content, such as images, music, text, or videos, based on patterns and algorithms learned from existing data. This AI can generate new content that is similar to, but not identical to, existing content, making it a powerful tool for creative industries, content creation, and even art.

• **Unsupervised learning**: Generative AI models are typically trained using unsupervised learning, which means they are trained on large datasets without labeled or annotated data. This allows the AI to learn patterns and relationships in the data, and then use these patterns to generate new content that is similar to, but distinct from, the original data.


## Get Grok API

Here you need to get an access token to be able to access models using Grok's platform via APIs:

- Groq API Key: Go [here](https://console.groq.com/keys) and create an API key. You need to setup an account which is totally free of cost. Also while Groq has a generous free tier, there are also paid plans if you are interested.


1. Go to [Groq Cloud -> Create API Key](https://console.groq.com/keys) after creating your account and make sure to create a new API Key as shown

![](https://i.imgur.com/tgHXlcV.png)

2. Remember to __Save__ your key somewhere safe as it will just be shown once as shown below. So copy and save it in a local secure file to use it later on. If you forget, just create a new key anytime.

![](https://i.imgur.com/Q27AgA1.png)

## Load Groq API Credentials


In [18]:
from getpass import getpass

groq_key = getpass("Enter your Groq API Key: ")

Enter your Groq API Key:  ········


## Using Open Source LLMs Directly via Groq API

This is if you want to use it without wrappers like LangChain, we will show you how you use open LLMs like Meta Llama 3.2 Instruct using Groq APIs. The free tier should be good enough for most experiments.

## API Pricing

Right now the best models to use include Mistral, Gemma 2 and Llama 3.1 and 3.2. Check out [pricing details here for free API](https://console.groq.com/settings/limits) and [here for paid API](https://groq.com/pricing/)

![](https://i.imgur.com/JE8lfXV.png)

## Use Groq for Prompting Open Source LLMs

In [19]:
from groq import Groq

groq_client = Groq(api_key=groq_key)

In [20]:
def get_completion_chatgroq(prompt, model="meta-llama/llama-guard-4-12b"):
    messages = [{"role": "user", "content": prompt}]
    response = groq_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # degree of randomness of the model's output
    )
    return response.choices[0].message.content

In [21]:
# models keep getting updated and older ones get deprecated
# refer to latest list here: https://console.groq.com/dashboard/limits
prompt = 'Explain Generative AI in 2 bullet points'
response = get_completion_chatgroq(prompt=prompt, model="meta-llama/llama-4-scout-17b-16e-instruct")

print(response)

Here are 2 bullet points explaining Generative AI:

* **Creates new content**: Generative AI is a type of artificial intelligence that generates new, original content, such as images, videos, music, text, and more. It uses complex algorithms and machine learning techniques to create novel outputs that are similar in style and structure to existing data.
* **Learns from existing data**: Generative AI models are trained on large datasets of existing content, which allows them to learn patterns, relationships, and characteristics of the data. This training enables the AI to generate new content that is coherent, realistic, and often indistinguishable from human-created content.


In [22]:
prompt = 'Explain Generative AI in 2 bullet points'
response = get_completion_chatgroq(prompt=prompt, model="llama-3.3-70b-versatile")

print(response)

* **Definition and Function**: Generative AI refers to a type of artificial intelligence that uses machine learning algorithms to generate new, original content, such as images, videos, music, text, or other forms of data. This is achieved by training the AI model on a large dataset, allowing it to learn patterns and relationships within the data, and then using this knowledge to create new, synthetic data that resembles the original.
* **Applications and Capabilities**: Generative AI has a wide range of applications, including art and design, music and audio generation, text-to-image synthesis, and data augmentation. It can be used to create realistic images and videos, generate new music and audio tracks, produce coherent and context-specific text, and even create new products and designs. The capabilities of generative AI are constantly evolving, and it has the potential to revolutionize various industries, from entertainment and media to healthcare and education.
